## Graph RAG - 관계형 검색 원리

일반적인 벡터 기반 RAG는 질문과 "의미적으로 유사한" 텍스트 조각을 찾아 컨텍스트로 사용한다.
그런데 아래와 같은 질문에는 벡터 유사도만으로 답하기 어렵다.

> "김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?"

이 질문에 답하려면 **김민준 → 그가 맡은 프로젝트 → 그 프로젝트가 의존하는 다른 프로젝트 → 그 프로젝트의 담당자**
로 이어지는 여러 단계(멀티홉, multi-hop)의 **관계**를 따라가야 한다. 개별 문장은 질문과 표현이 겹치지 않을 수 있어,
유사도 검색만으로는 필요한 사실들을 한 번에 모으기 어렵다.

**Graph RAG**는 문서에서 (주체, 관계, 객체) 형태의 **트리플(triple)**을 추출해 지식 그래프(Knowledge Graph)로
저장하고, 질문에 등장하는 개체(entity)에서 출발해 그래프를 **순회(traversal)**하면서 관련된 사실들을 모아
컨텍스트를 구성하는 방식이다.

### 흐름
1. 비정형 텍스트에서 LLM으로 (주체, 관계, 객체) 트리플 추출
2. 트리플을 그래프(노드=개체, 엣지=관계)로 저장
3. 질문에서 개체를 인식하고, 그래프를 1~k홉 순회하며 관련 서브그래프 수집
4. 서브그래프를 텍스트 컨텍스트로 변환해 LLM이 최종 답변 생성

본 노트북에서는 **Naive Vector RAG의 한계**를 먼저 확인한 뒤, **트리플 추출 → 그래프 구축 → 그래프 순회 검색**으로
이어지는 Graph RAG 파이프라인을 직접 구현하고 두 방식을 비교한다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

### [0] 공통 준비: LLM, 임베딩 모델

In [2]:
from typing import List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_4060\630418706.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### [1] 샘플 도메인: 테크노바 조직·프로젝트 지식베이스

가상의 회사 "테크노바"의 조직 구성과 프로젝트 담당 관계를 사실 문장(fact) 단위로 정리했다.
각 문장은 하나의 관계(트리플)를 담고 있으며, 뒤에서 이 문장들을 그대로 벡터 검색용 문서로도 쓰고
그래프용 트리플 추출의 원문으로도 사용한다.

In [3]:
# 관계 어휘(controlled vocabulary): MEMBER_OF, LEADS, OWNS, COLLABORATES_WITH, DEPENDS_ON, WORKS_ON
company_facts = [
    "김민준은 테크노바의 AI팀 소속이다.",
    "이서연은 AI팀의 팀장이다.",
    "AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.",
    "박지훈은 데이터팀 소속이다.",
    "최유진은 데이터팀의 팀장이다.",
    "데이터팀은 '추천시스템 고도화' 프로젝트를 담당한다.",
    "AI팀은 데이터팀과 긴밀히 협업한다.",
    "정다은은 인프라팀 소속이다.",
    "한소희는 인프라팀의 팀장이다.",
    "인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.",
    "데이터팀은 인프라팀과 긴밀히 협업한다.",
    "'그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.",
    "박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.",
    "김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.",
    "프로덕트팀은 '온보딩 자동화' 프로젝트를 담당한다.",
    "정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.",
]

docs = [
    Document(page_content=fact, metadata={"fact_id": i})
    for i, fact in enumerate(company_facts)
]

vectorstore = FAISS.from_documents(docs, embeddings)
print(f"사실 문장 수: {len(company_facts)}")

사실 문장 수: 16


In [4]:
docs

[Document(metadata={'fact_id': 0}, page_content='김민준은 테크노바의 AI팀 소속이다.'),
 Document(metadata={'fact_id': 1}, page_content='이서연은 AI팀의 팀장이다.'),
 Document(metadata={'fact_id': 2}, page_content="AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다."),
 Document(metadata={'fact_id': 3}, page_content='박지훈은 데이터팀 소속이다.'),
 Document(metadata={'fact_id': 4}, page_content='최유진은 데이터팀의 팀장이다.'),
 Document(metadata={'fact_id': 5}, page_content="데이터팀은 '추천시스템 고도화' 프로젝트를 담당한다."),
 Document(metadata={'fact_id': 6}, page_content='AI팀은 데이터팀과 긴밀히 협업한다.'),
 Document(metadata={'fact_id': 7}, page_content='정다은은 인프라팀 소속이다.'),
 Document(metadata={'fact_id': 8}, page_content='한소희는 인프라팀의 팀장이다.'),
 Document(metadata={'fact_id': 9}, page_content="인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다."),
 Document(metadata={'fact_id': 10}, page_content='데이터팀은 인프라팀과 긴밀히 협업한다.'),
 Document(metadata={'fact_id': 11}, page_content="'그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다."),
 Document(metadata={'fact_id': 12}, page_content="박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다."),
 Docume

### [2] Naive Vector RAG의 한계: 멀티홉 질문 실험

질문을 그대로 임베딩해 유사한 사실 문장을 검색한 뒤 답변을 생성한다.

In [5]:
def format_docs(documents: List[Document]) -> str:
    return "\n".join(f"- {d.page_content}" for d in documents)


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [사실] 목록만 근거로 질문에 답하라. 목록에 없는 내용은 추측하지 말고 "
            "'주어진 사실만으로는 알 수 없다'라고 답하라.\n\n[사실]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
answer_chain = answer_prompt | llm | StrOutputParser()


def naive_rag(question: str, k: int = 4) -> dict:
    retrieved = vectorstore.similarity_search(question, k=k)
    answer = answer_chain.invoke(
        {"context": format_docs(retrieved), "question": question}
    )
    return {"retrieved": retrieved, "answer": answer}


query_1 = "김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?"

naive_result = naive_rag(query_1)
print("=== Naive RAG 검색 결과 ===")
print(format_docs(naive_result["retrieved"]))
print("\n=== Naive RAG 답변 ===")
print(naive_result["answer"])

=== Naive RAG 검색 결과 ===
- 김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.
- 박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.
- 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
- 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.

=== Naive RAG 답변 ===
주어진 사실만으로는 알 수 없다.


질문을 "김민준"·"프로젝트"·"의존" 등의 표현과 의미적으로 가까운 문장 위주로 검색하다 보니,
정답까지 이어지는 3단계 연결고리(김민준 → 그래프 RAG 엔진 → 검색 인프라 개선 → 정다은) 중 일부가
top-k 안에 들어오지 않거나, 들어오더라도 LLM이 사실들을 순서대로 연결해야 한다는 부담이 그대로 남는다.
`k`를 늘리면 우연히 맞힐 수도 있지만, 코퍼스가 커질수록 신뢰할 수 없는 방식이다.

### [3] 지식 그래프 구축 (1) — LLM으로 트리플 추출

비정형 문장에서 **(주체, 관계, 객체)** 구조를 뽑아낸다. `with_structured_output`으로 Pydantic 스키마를
강제하면 파싱 없이 바로 트리플 리스트를 얻을 수 있다.

In [6]:
from pydantic import BaseModel, Field


class Triple(BaseModel):
    subject: str = Field(description="관계의 주체가 되는 개체명")
    relation: str = Field(
        description=(
            "MEMBER_OF, LEADS, OWNS, COLLABORATES_WITH, DEPENDS_ON, WORKS_ON 중 하나"
        )
    )
    object: str = Field(description="관계의 대상이 되는 개체명")


class TripleList(BaseModel):
    triples: List[Triple]


triple_extractor = llm.with_structured_output(TripleList)

extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "다음 문장들에서 (주체, 관계, 객체) 트리플을 모두 추출하라.\n"
            "- relation은 반드시 MEMBER_OF, LEADS, OWNS, COLLABORATES_WITH, DEPENDS_ON, WORKS_ON 중 하나로 매핑한다.\n"
            "- 개체명은 문장에 등장한 표기를 그대로 사용하고, 작은따옴표는 제거한다.\n"
            "- 한 문장에 여러 관계가 있으면 모두 추출한다.",
        ),
        ("human", "{text}"),
    ]
)
extract_chain = extract_prompt | triple_extractor

source_text = "\n".join(company_facts)
extracted = extract_chain.invoke({"text": source_text})

triples = [(t.subject, t.relation, t.object) for t in extracted.triples]

print(f"추출된 트리플 수: {len(triples)}")
for s, r, o in triples:
    print(f"({s}) -[{r}]-> ({o})")

추출된 트리플 수: 16
(김민준) -[MEMBER_OF]-> (테크노바의 AI팀)
(이서연) -[LEADS]-> (AI팀)
(AI팀) -[WORKS_ON]-> (그래프 RAG 엔진)
(박지훈) -[MEMBER_OF]-> (데이터팀)
(최유진) -[LEADS]-> (데이터팀)
(데이터팀) -[WORKS_ON]-> (추천시스템 고도화)
(AI팀) -[COLLABORATES_WITH]-> (데이터팀)
(정다은) -[MEMBER_OF]-> (인프라팀)
(한소희) -[LEADS]-> (인프라팀)
(인프라팀) -[WORKS_ON]-> (검색 인프라 개선)
(데이터팀) -[COLLABORATES_WITH]-> (인프라팀)
(그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
(박지훈) -[WORKS_ON]-> (추천시스템 고도화)
(김민준) -[WORKS_ON]-> (그래프 RAG 엔진)
(프로덕트팀) -[WORKS_ON]-> (온보딩 자동화)
(정다은) -[WORKS_ON]-> (검색 인프라 개선)


### [4] 지식 그래프 구축 (2) — NetworkX 그래프에 저장

추출된 트리플을 방향 그래프(`MultiDiGraph`)의 엣지로 저장한다. 노드=개체, 엣지 라벨=관계 타입이다.

In [8]:
triples

[('김민준', 'MEMBER_OF', '테크노바의 AI팀'),
 ('이서연', 'LEADS', 'AI팀'),
 ('AI팀', 'WORKS_ON', '그래프 RAG 엔진'),
 ('박지훈', 'MEMBER_OF', '데이터팀'),
 ('최유진', 'LEADS', '데이터팀'),
 ('데이터팀', 'WORKS_ON', '추천시스템 고도화'),
 ('AI팀', 'COLLABORATES_WITH', '데이터팀'),
 ('정다은', 'MEMBER_OF', '인프라팀'),
 ('한소희', 'LEADS', '인프라팀'),
 ('인프라팀', 'WORKS_ON', '검색 인프라 개선'),
 ('데이터팀', 'COLLABORATES_WITH', '인프라팀'),
 ('그래프 RAG 엔진', 'DEPENDS_ON', '검색 인프라 개선'),
 ('박지훈', 'WORKS_ON', '추천시스템 고도화'),
 ('김민준', 'WORKS_ON', '그래프 RAG 엔진'),
 ('프로덕트팀', 'WORKS_ON', '온보딩 자동화'),
 ('정다은', 'WORKS_ON', '검색 인프라 개선')]

In [9]:
import networkx as nx

graph = nx.MultiDiGraph()

for subject, relation, obj in triples:
    graph.add_edge(subject, obj, relation=relation)

print(f"노드 수: {graph.number_of_nodes()}")
print(f"엣지 수: {graph.number_of_edges()}")
print("\n노드 목록:")
print(sorted(graph.nodes()))

노드 수: 15
엣지 수: 16

노드 목록:
['AI팀', '검색 인프라 개선', '그래프 RAG 엔진', '김민준', '데이터팀', '박지훈', '온보딩 자동화', '이서연', '인프라팀', '정다은', '최유진', '추천시스템 고도화', '테크노바의 AI팀', '프로덕트팀', '한소희']


### [5] 그래프 순회 기반 관계형 검색 (Multi-hop)

1. 질문 문자열 안에 등장하는 그래프 노드를 찾아 **시작 개체(seed)**로 삼는다.
   (실제 서비스에서는 NER/엔티티 링킹을 쓰지만, 여기서는 노드명이 질문에 그대로 포함되는지로 단순화한다.)
2. seed에서 출발해 **양방향(들어오는/나가는 엣지)**으로 최대 k홉까지 확장하며 지나온 엣지를 모두 수집한다.
3. 수집한 엣지를 `주체 -[관계]-> 객체` 텍스트로 변환해 컨텍스트로 사용한다.

In [10]:
def find_seed_entities(question: str, g: nx.MultiDiGraph) -> List[str]:
    '''질문 문자열에 노드명이 부분 문자열로 포함되면 시작 개체로 채택한다.'''
    return [node for node in g.nodes() if node in question]


def k_hop_edges(g: nx.MultiDiGraph, seeds: List[str], k: int = 3) -> List[tuple]:
    '''seed로부터 양방향으로 최대 k홉까지 순회하며 지나온 엣지를 수집한다.'''
    visited_nodes = set(seeds)
    frontier = set(seeds)
    collected_edges = set()

    for _ in range(k):
        next_frontier = set()
        for node in frontier:
            # 나가는 엣지: node -[relation]-> neighbor
            for _, neighbor, data in g.out_edges(node, data=True):
                collected_edges.add((node, data["relation"], neighbor))
                if neighbor not in visited_nodes:
                    next_frontier.add(neighbor)
            # 들어오는 엣지: neighbor -[relation]-> node
            for neighbor, _, data in g.in_edges(node, data=True):
                collected_edges.add((neighbor, data["relation"], node))
                if neighbor not in visited_nodes:
                    next_frontier.add(neighbor)
        visited_nodes |= next_frontier
        frontier = next_frontier
        if not frontier:
            break

    return sorted(collected_edges)


def edges_to_context(edges: List[tuple]) -> str:
    return "\n".join(f"- ({s}) -[{r}]-> ({o})" for s, r, o in edges)


def graph_retrieve(question: str, g: nx.MultiDiGraph, k: int = 3) -> dict:
    seeds = find_seed_entities(question, g)
    edges = k_hop_edges(g, seeds, k=k) if seeds else []
    return {"seeds": seeds, "edges": edges, "context": edges_to_context(edges)}


retrieval_1 = graph_retrieve(query_1, graph, k=3)
print(f"시작 개체(seed): {retrieval_1['seeds']}")
print("\n=== 그래프 순회로 수집한 관계 ===")
print(retrieval_1["context"])

시작 개체(seed): ['김민준']

=== 그래프 순회로 수집한 관계 ===
- (AI팀) -[COLLABORATES_WITH]-> (데이터팀)
- (AI팀) -[WORKS_ON]-> (그래프 RAG 엔진)
- (그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
- (김민준) -[MEMBER_OF]-> (테크노바의 AI팀)
- (김민준) -[WORKS_ON]-> (그래프 RAG 엔진)
- (이서연) -[LEADS]-> (AI팀)
- (인프라팀) -[WORKS_ON]-> (검색 인프라 개선)
- (정다은) -[WORKS_ON]-> (검색 인프라 개선)


수집된 서브그래프를 컨텍스트로 넘겨 답변을 생성한다.

In [11]:
graph_answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [관계] 목록은 지식 그래프에서 질문과 관련해 순회로 수집한 사실이다. "
            "이 관계들을 연결해 질문에 답하라. 목록만으로 답할 수 없으면 '알 수 없다'라고 답하라.\n\n"
            "[관계]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
graph_answer_chain = graph_answer_prompt | llm | StrOutputParser()


def graph_rag(question: str, g: nx.MultiDiGraph, k: int = 3) -> dict:
    retrieval = graph_retrieve(question, g, k=k)
    answer = graph_answer_chain.invoke(
        {"context": retrieval["context"], "question": question}
    )
    return {**retrieval, "answer": answer}


graph_result_1 = graph_rag(query_1, graph, k=3)
print("=== Graph RAG 답변 ===")
print(graph_result_1["answer"])

=== Graph RAG 답변 ===
김민준이 담당하는 프로젝트인 그래프 RAG 엔진은 검색 인프라 개선에 의존하고 있습니다. 검색 인프라 개선은 인프라팀과 정다은이 담당하고 있습니다.


### [6] Naive Vector RAG vs Graph RAG 비교

멀티홉 추론이 필요한 질문 2개로 두 방식을 나란히 비교한다.

In [13]:
query_2 = "이서연이 이끄는 팀과 협업하는 팀이 담당하는 프로젝트는?"

test_queries = [query_1, query_2]

for q in test_queries:
    naive = naive_rag(q)
    g_rag = graph_rag(q, graph, k=3)

    print(f"질문: {q}")
    print(f"- Naive Vector RAG: {naive['answer']}")
    print(f"- Graph RAG        : {g_rag['answer']}")
    print("-" * 80)

질문: 김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?
- Naive Vector RAG: 주어진 사실만으로는 알 수 없다.
- Graph RAG        : 김민준이 담당하는 프로젝트인 그래프 RAG 엔진은 검색 인프라 개선에 의존하고 있습니다. 검색 인프라 개선은 인프라팀과 정다은이 담당하고 있습니다.
--------------------------------------------------------------------------------
질문: 이서연이 이끄는 팀과 협업하는 팀이 담당하는 프로젝트는?
- Naive Vector RAG: 주어진 사실만으로는 알 수 없다.
- Graph RAG        : 이서연이 이끄는 AI팀은 데이터팀과 협업하고 있습니다. 데이터팀이 담당하는 프로젝트는 추천시스템 고도화입니다.
--------------------------------------------------------------------------------


두 질문 모두 정답에 필요한 사실이 서로 다른 문장 여러 개에 흩어져 있고, 질문 표현과
직접적으로 겹치지 않는 문장(예: "정다은은 '검색 인프라 개선' 프로젝트의 담당자이다")이 중간 연결고리다.
Graph RAG는 seed 개체에서 출발해 관계를 순서대로 따라가므로 이런 연결을 빠뜨리지 않지만,
Naive Vector RAG는 top-k 유사도에 의존하기 때문에 연결고리 문장이 누락되면 답을 만들 수 없다.

### [7] 실제 서비스에서는 어떻게 확장하나

이 노트북은 원리를 보여주기 위해 NetworkX 인메모리 그래프와 문자열 매칭 기반 엔티티 인식을 사용했다.
실제 프로덕션 Graph RAG 파이프라인에서는 보통 다음 구성 요소를 사용한다.

- **그래프 자동 구축**: `langchain_experimental.graph_transformers.LLMGraphTransformer`로 대량의 비정형
  문서에서 `GraphDocument`(노드/관계)를 자동 추출한다. (`pip install langchain-experimental`)
- **그래프 저장소**: Neo4j 같은 그래프 DB(`langchain_community.graphs.Neo4jGraph`)에 저장하면 대규모
  그래프에서도 인덱스를 활용한 빠른 순회가 가능하다.
- **질의**: `GraphCypherQAChain`처럼 LLM이 질문을 그래프 쿼리 언어(Cypher)로 변환해 그래프 DB에 직접
  질의하는 방식도 널리 쓰인다.
- **엔티티 인식**: 이 노트북의 부분 문자열 매칭 대신, NER 모델이나 LLM 기반 엔티티 링킹으로 질문 속
  개체를 식별한다.
- **홉 수 결정**: 이 노트북은 `k=3`을 고정값으로 썼지만, 질문마다 필요한 홉 수는 다르다. 이후 노트북에서
  다룰 **자율 검색 에이전트(agentic retrieval)**는 각 홉마다 "더 순회할지, 답변할지"를 LLM이 스스로
  판단하도록 확장한 형태다.

```python
# 참고: LLMGraphTransformer 예시 (실행에는 langchain-experimental 및 그래프 스키마 설계 필요)
from langchain_experimental.graph_transformers import LLMGraphTransformer

graph_transformer = LLMGraphTransformer(llm=llm)
graph_documents = graph_transformer.convert_to_graph_documents(docs)
```

### [8] 정리

| 구분 | Naive Vector RAG | Graph RAG |
|---|---|---|
| 검색 단위 | 질문과 유사한 텍스트 조각 | 개체(entity)에서 출발한 관계 서브그래프 |
| 강점 | 구현이 단순, 의미적 유사 문서 검색에 강함 | 멀티홉 추론, 개체 간 명시적 관계 추적에 강함 |
| 약점 | 여러 사실을 연결해야 하는 질문에 취약 | 그래프 구축 비용, 엔티티 인식 정확도에 의존 |
| 컨텍스트 근거 | top-k 유사 문서 | seed로부터 k홉 이내의 관계 트리플 |

**구현 포인트**
- 트리플 추출: `llm.with_structured_output(TripleList)`로 (주체, 관계, 객체) 구조화 추출
- 그래프 저장: `networkx.MultiDiGraph` — 노드=개체, 엣지 속성=관계 타입
- 그래프 검색: 질문에서 seed 개체 인식 → 양방향 k홉 순회 → 수집된 트리플을 컨텍스트로 변환

**언제 쓰면 좋은가**
- "A와 B는 어떤 관계인가?", "A가 속한 X의 Y는 무엇인가?"처럼 개체 간 관계·경로를 묻는 질문
- 정답 근거가 하나의 문서/문장이 아니라 여러 사실을 연결해야 나오는 경우
- 벡터 검색과 결합해, 벡터 검색으로 후보 개체를 찾고 그래프로 관계를 확장하는 하이브리드 구성도 흔히 쓰인다